# Sanity-check разметки интентов

**Цель ноутбука.** Выполнить базовый sanity-check ручной разметки интентов русскоязычных диалогов из датасета `d0rj/dialogsum-ru` в рамках магистерской диссертации «Семантический анализ русскоязычных диалогов для задачи распознавания намерений с улучшением на базе предобученных моделей».

**Что делает ноутбук:**

- подключает Google Drive в среде Google Colab;
- загружает размеченный CSV `dialogue_intent_annotation_1246_v1_annotated.csv` с Google Drive;
- считает общее число размеченных диалогов;
- анализирует распределение значений `primary_intent`;
- проверяет дисбаланс классов;
- сохраняет таблицу распределения в `results/tables` и график в `results/figures` на Google Drive.

Ноутбук **не обучает модели** и **не модифицирует исходный размеченный CSV**.
Ноутбук рассчитан на запуск в Google Colab.

## 1. Подключение Google Drive и пути к артефактам

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/russian-dialogue-intent-thesis"
DATA_DIR = f"{BASE_DIR}/data"
ANNO_DIR = f"{DATA_DIR}/annotation"
RESULTS_DIR = f"{BASE_DIR}/results"
TABLES_DIR = f"{RESULTS_DIR}/tables"
FIGURES_DIR = f"{RESULTS_DIR}/figures"

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"ANNO_DIR:    {ANNO_DIR}")
print(f"TABLES_DIR:  {TABLES_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")

## 2. Импорт библиотек

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 3. Загрузка размеченного CSV

Файл с ручной разметкой лежит на Google Drive в `data/annotation/`. Ноутбук только читает этот файл и **не вносит в него изменений**.

In [ ]:
ANNOTATED_CSV = f"{ANNO_DIR}/dialogue_intent_annotation_1246_v1_annotated.csv"

df = pd.read_csv(ANNOTATED_CSV)

print(f"Число строк: {len(df)}")
print(f"Столбцы: {list(df.columns)}")
df.head()

## 4. Подсчёт общего числа размеченных диалогов

In [ ]:
N = len(df)

# Считаем строки с непустым primary_intent
if "primary_intent" not in df.columns:
    raise KeyError("В CSV отсутствует столбец 'primary_intent' — проверьте файл разметки.")

non_empty_mask = df["primary_intent"].notna() & (df["primary_intent"].astype(str).str.strip() != "")
n_annotated = int(non_empty_mask.sum())
n_missing = N - n_annotated

print(f"Всего строк в файле:                 {N}")
print(f"Размечено (непустой primary_intent): {n_annotated}")
print(f"Без разметки primary_intent:         {n_missing}")

## 5. Распределение классов `primary_intent`

Считаем количество и долю каждого класса, сортируем по убыванию частоты и сохраняем таблицу в `results/tables/primary_intent_distribution.csv`.

In [ ]:
counts = df.loc[non_empty_mask, "primary_intent"].value_counts()

dist_df = (
    counts.rename("count")
    .reset_index()
    .rename(columns={"index": "primary_intent"})
)
# В новых pandas value_counts().reset_index() уже даёт колонку 'primary_intent';
# на всякий случай нормализуем имена столбцов:
if dist_df.columns[0] != "primary_intent":
    dist_df.columns = ["primary_intent", "count"]

dist_df["proportion"] = dist_df["count"] / dist_df["count"].sum()
dist_df = dist_df.sort_values("count", ascending=False).reset_index(drop=True)

dist_csv_path = f"{TABLES_DIR}/primary_intent_distribution.csv"
dist_df.to_csv(dist_csv_path, index=False)

print(f"Таблица сохранена: {dist_csv_path}")
dist_df

## 6. Визуализация распределения

Строим барплот по числу примеров на класс, подписываем столбцы количеством и долей, сохраняем PNG в `results/figures/primary_intent_distribution.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(dist_df)), 6))

sns.barplot(
    data=dist_df,
    x="primary_intent",
    y="count",
    order=dist_df["primary_intent"].tolist(),
    color="#4C72B0",
    ax=ax,
)

ax.set_title("Распределение классов primary_intent")
ax.set_xlabel("primary_intent")
ax.set_ylabel("Количество диалогов")

for label in ax.get_xticklabels():
    label.set_rotation(45)
    label.set_horizontalalignment("right")

# Подписи над столбцами: count (доля)
for patch, (_, row) in zip(ax.patches, dist_df.iterrows()):
    height = patch.get_height()
    ax.annotate(
        f"{int(row['count'])}\n({row['proportion']:.1%})",
        xy=(patch.get_x() + patch.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()

fig_path = f"{FIGURES_DIR}/primary_intent_distribution.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"График сохранён: {fig_path}")

## 7. Текстовый отчёт о дисбалансе классов

Печатаем краткую сводку: самый частый класс, классы с малой частотой (`count < 10`), классы с долей выше 0.4 и 0.5, и рекомендацию по метрикам.

In [ ]:
top_row = dist_df.iloc[0]
top_class = top_row["primary_intent"]
top_prop = float(top_row["proportion"])
top_count = int(top_row["count"])

rare_classes = dist_df[dist_df["count"] < 10]
high_04 = dist_df[dist_df["proportion"] > 0.4]
high_05 = dist_df[dist_df["proportion"] > 0.5]

min_count = int(dist_df["count"].min()) if len(dist_df) else 0
max_prop = float(dist_df["proportion"].max()) if len(dist_df) else 0.0

print("=== Краткий отчёт по распределению primary_intent ===")
print(f"Размечено диалогов:          {n_annotated}")
print(f"Число уникальных классов:    {len(dist_df)}")
print(f"Самый частый класс:          {top_class} (count={top_count}, доля={top_prop:.1%})")
print(f"Минимальный count по классу: {min_count}")
print(f"Максимальная доля по классу: {max_prop:.1%}")

print()
print(f"Классы с count < 10 (всего {len(rare_classes)}):")
if len(rare_classes):
    for _, row in rare_classes.iterrows():
        print(f"  - {row['primary_intent']}: count={int(row['count'])}, доля={row['proportion']:.1%}")
else:
    print("  (нет)")

print()
print(f"Классы с долей > 0.4 (всего {len(high_04)}):")
if len(high_04):
    for _, row in high_04.iterrows():
        print(f"  - {row['primary_intent']}: доля={row['proportion']:.1%}")
else:
    print("  (нет)")

print()
print(f"Классы с долей > 0.5 (всего {len(high_05)}):")
if len(high_05):
    for _, row in high_05.iterrows():
        print(f"  - {row['primary_intent']}: доля={row['proportion']:.1%}")
else:
    print("  (нет)")

print()
print("=== Рекомендация ===")
imbalanced = (max_prop > 0.4) or (min_count < 10)
if imbalanced:
    print(
        "Датасет существенно несбалансирован (есть класс с долей > 40% "
        "и/или класс с count < 10).\n"
        "Для многоклассовой классификации рекомендуется использовать macro-F1 "
        "как основную метрику и применять веса классов (class weights) "
        "или иные техники компенсации дисбаланса при обучении."
    )
else:
    print(
        "Датасет приблизительно сбалансирован (нет класса с долей > 40% "
        "и минимальный count >= 10).\n"
        "Тем не менее, для многоклассовой классификации рекомендуется "
        "наряду с accuracy сообщать macro-F1."
    )

print("\nВнимание: оценки метрик моделей в этом ноутбуке НЕ считаются — "
      "только распределение разметки.")

## 8. Анализ неоднозначных размеченных диалогов (`is_ambiguous`)

Если в файле разметки присутствует столбец `is_ambiguous`, считаем число диалогов с `is_ambiguous == 1` и смотрим, какие классы `primary_intent` среди них встречаются чаще всего.

In [ ]:
if "is_ambiguous" in df.columns:
    amb_series = pd.to_numeric(df["is_ambiguous"], errors="coerce")
    amb_mask = (amb_series == 1) & non_empty_mask
    n_ambiguous = int(amb_mask.sum())

    print(f"Число строк с is_ambiguous == 1 (и непустым primary_intent): {n_ambiguous}")

    if n_ambiguous > 0:
        amb_counts = (
            df.loc[amb_mask, "primary_intent"]
            .value_counts()
            .rename("count")
            .reset_index()
        )
        if amb_counts.columns[0] != "primary_intent":
            amb_counts.columns = ["primary_intent", "count"]
        amb_counts["proportion_within_ambiguous"] = amb_counts["count"] / amb_counts["count"].sum()

        print("\nРаспределение primary_intent среди неоднозначных диалогов:")
        display(amb_counts)
    else:
        print("Неоднозначных размеченных диалогов нет.")
else:
    print("Столбец 'is_ambiguous' отсутствует — раздел пропускается.")

## 9. Итог: созданные файлы и интерпретация

**При запуске ноутбука на Google Drive создаются файлы:**

- `results/tables/primary_intent_distribution.csv` — таблица распределения классов `primary_intent` (`primary_intent`, `count`, `proportion`).
- `results/figures/primary_intent_distribution.png` — барплот распределения классов.

Артефакты сохраняются на Google Drive и не коммитятся в репозиторий.

**Как интерпретировать результаты:**

- Если один класс занимает > 40–50% разметки, либо для каких-то классов `count < 10`, дисбаланс считается существенным. В таком случае при последующем обучении классификаторов имеет смысл использовать `class_weight`/взвешенные функции потерь и опираться на macro-F1, а не на accuracy.
- Если распределение более равномерное и редких классов нет, дисбаланс несущественный, но для многоклассовой задачи всё равно желательно отчитываться по macro-F1 наряду с accuracy.
- Большое число неоднозначных диалогов (`is_ambiguous == 1`), сконцентрированных в конкретных классах, может указывать на необходимость уточнения инструкции по разметке или пересмотра границ между этими интентами.

Конкретные числовые выводы по текущей версии разметки см. в выводе ячейки с текстовым отчётом выше.